# BlisterTrace — Submission Vision Candidate Training

**Real defects, not the earlier single-class starter.** Target: Pi 5 16 GB + Hailo-8 26 TOPS.

## Before starting
1. Save your previous notebook outputs. Import this notebook into a fresh Kaggle notebook.
2. Select GPU T4 and enable Internet. One GPU is used initially for reliable notebook execution.
3. Create a Roboflow account and find your **private API key** at [Roboflow settings](https://app.roboflow.com/settings/api). In Kaggle Secrets, add `ROBOFLOW_API_KEY` and grant this notebook access. **Never paste the key into chat or notebook code.**
4. Run Steps 1–6. Review the defect labels and grouping results before enabling training.
5. Set `DATA_REVIEWED=True` in Step 3, rerun Step 3, then continue from Step 7.

## What is downloaded and what it covers
The public [Blister Strips defect dataset](https://universe.roboflow.com/blister-strips/blister-strips-defect-detection) lists CC BY 4.0. The downloaded version 2 declares three defect classes (`broken`, `cracked`, `missing`). A `strip` context class is optional and used only if declared in the actual export. Its model page documents version 2 with 1,505 exported images; that count may include augmentation. **Counts and labels are checked after download, not assumed.** Other versions are not silently substituted.

This source does **not** establish support for discoloration, wrong contents, foreign particles, foil damage, or arbitrary capsule layouts. Those need additional labeled data. Publicly reported model scores are not this project's results.

This notebook produces a reproducible **vision candidate**, held-out evaluation, validation-selected thresholds, a model contract, a static ONNX with FP32 parity checks, training-only calibration images, and a submission evidence bundle.

**No notebook can guarantee a perfect or winning model.** Hailo INT8 compilation/quality, physical Pi measurements, and mandatory Llama/ExecuTorch advisory are separate checks. Missing evidence is recorded explicitly. The notebook never turns a GPU timing into Pi FPS or claims an ONNX file is a tested HEF.


## Step 1 — Verify the known Kaggle environment
The previous starter worked on Python 3.13, PyTorch 2.11 and torchvision 0.26. Keep the working training version instead of upgrading during an experiment.


In [ ]:
import os, sys, json, subprocess, platform
from pathlib import Path
import torch
WORK=Path('/kaggle/working/blistertrace_submission')
WORK.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Select GPU T4.'
print('Python:',sys.version.split()[0], 'Torch:',torch.__version__)
for index in range(torch.cuda.device_count()):
    print(index,torch.cuda.get_device_name(index))
print('Training uses GPU 0. This is host validation, not a Pi benchmark.')


## Step 2 — Install pinned training and export dependencies
These are separate from any future ExecuTorch environment. The exporter explicitly uses the legacy static ONNX path supported by the PyTorch API, with opset 11 from the Hailo-8 YOLOv8 recipe. The exact custom model still requires vendor-parser validation.


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'ultralytics==8.3.161','roboflow-slim==1.7.3','pillow-avif-plugin==1.6.0',
    'onnx==1.19.1','onnxruntime==1.23.2'])
import torchvision, ultralytics, onnx, onnxruntime
from ultralytics import YOLO
for device in ('cpu','cuda:0'):
    boxes=torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]],device=device)
    scores=torch.tensor([0.9,0.8],device=device)
    print(device,torchvision.ops.nms(boxes,scores,0.5).tolist())
print('Ultralytics:',ultralytics.__version__)
print('ONNX:',onnx.__version__,'ORT:',onnxruntime.__version__)
try:
    import roboflow, pillow_avif
except ModuleNotFoundError as error:
    missing=error.name or 'unknown'
    if not missing.replace('_','').replace('.','').isalnum(): missing='unknown'
    raise RuntimeError('Roboflow dependency missing: '+missing+'. Resolve this before Step 4.') from None
print('Roboflow import passed.')
(WORK/'environment.txt').write_text(subprocess.check_output(
    [sys.executable,'-m','pip','freeze'],text=True))


## Embedded helper code
This cell contains the readable helper module so **only one .ipynb upload is needed**. It defines data checks, leakage grouping, held-out splitting, detection matching, threshold selection, and export checks. It performs no training or network access by itself.


In [ ]:
HELPER_SOURCE = '"""Reproducible blister-defect training helpers, embedded in the Kaggle notebook.\n\nNo secrets, model downloads, or training run on import. Physical target tests\nand the separate Llama/ExecuTorch component remain separate deployment gates.\n"""\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport re\nimport shutil\n\n\ndef write_json(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value, indent=2, default=str))\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef source_family(stem):\n    """Roboflow export siblings share the filename preceding .rf.<hash>.\n\n    This is a source-image family, NOT proof of physical-strip identity.\n    """\n    return re.split(r\'\\.rf\\.\', stem, maxsplit=1)[0]\n\n\ndef fingerprint(path):\n    import numpy as np\n    from PIL import Image\n    with Image.open(path) as opened:\n        rgb = opened.convert(\'RGB\')\n        rgb.load()\n        exact = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()\n        gray = np.asarray(rgb.convert(\'L\').resize((17, 16)), dtype=np.int16)\n        bits = (gray[:, 1:] > gray[:, :-1]).ravel()\n        perceptual = int.from_bytes(np.packbits(bits).tobytes(), \'big\')\n        return exact, perceptual, rgb.size\n\n\ndef load_boxes(path, class_count):\n    rows = []\n    if not Path(path).is_file():\n        raise ValueError(f\'Missing annotation: {path}\')\n    for line_no, line in enumerate(Path(path).read_text().splitlines(), 1):\n        if not line.strip():\n            continue\n        values = [float(v) for v in line.split()]\n        if len(values) != 5 or not all(math.isfinite(v) for v in values):\n            raise ValueError(f\'Expected five finite box values: {path}:{line_no}\')\n        c, x, y, w, h = values\n        if not c.is_integer() or not 0 <= c < class_count:\n            raise ValueError(f\'Invalid class: {path}:{line_no}\')\n        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):\n            raise ValueError(f\'Invalid box: {path}:{line_no}\')\n        if min(x-w/2, y-h/2) < -0.002 or max(x+w/2, y+h/2) > 1.002:\n            raise ValueError(f\'Out-of-bounds box: {path}:{line_no}\')\n        rows.append((int(c), x, y, w, h))\n    return rows\n\n\ndef compare_duplicate_annotations(first, second, size):\n    """Accept exact labels or same-class one-to-one matches within 1 px and IoU .99.\n\n    Applied only after decoded RGB pixel hashes match. The canonical annotation\n    is retained unchanged. Every rounded match includes both original labels.\n    """\n    if sorted(first) == sorted(second):\n        return {\'annotation_match\':\'exact\', \'max_corner_delta_pixels\':0., \'min_box_iou\':1.}\n    if len(first) != len(second):\n        return None\n    left = [xyxy(box,*size) for box in first]\n    right = [xyxy(box,*size) for box in second]\n    candidates = []\n    for a in left:\n        choices = []\n        for j,b in enumerate(right):\n            delta = max(abs(x-y) for x,y in zip(a[1:],b[1:]))\n            overlap = iou(a[1:],b[1:])\n            if a[0] == b[0] and delta <= 1.0+1e-9 and overlap >= .99-1e-12:\n                choices.append((j,delta,overlap))\n        candidates.append(choices)\n    matched = {}\n    def assign(index, seen):\n        for j,_,_ in candidates[index]:\n            if j in seen:\n                continue\n            seen.add(j)\n            if j not in matched or assign(matched[j],seen):\n                matched[j] = index\n                return True\n        return False\n    if not all(assign(i,set()) for i in range(len(left))):\n        return None\n    selected = [next((delta,overlap) for k,delta,overlap in candidates[i] if k==j)\n                for j,i in matched.items()]\n    return {\'annotation_match\':\'bounded_rounding\',\n            \'max_corner_delta_pixels\':max(d for d,_ in selected),\n            \'min_box_iou\':min(o for _,o in selected)}\n\n\ndef scan_export(root, names):\n    root = Path(root)\n    images = sorted(p for p in root.rglob(\'*\')\n                    if p.suffix.lower() in {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'}\n                    and p.parent.name == \'images\')\n    if not images:\n        raise ValueError(\'No images found in split/images folders.\')\n    records, exact_seen, duplicates = [], {}, []\n    for image in images:\n        label = image.parent.parent / \'labels\' / (image.stem + \'.txt\')\n        boxes = load_boxes(label, len(names))\n        exact, dhash, size = fingerprint(image)\n        if exact in exact_seen:\n            previous = exact_seen[exact]\n            comparison = compare_duplicate_annotations(previous[\'boxes\'],boxes,size)\n            if comparison is None:\n                raise ValueError(\n                    \'Identical image pixels have conflicting labels; review source. \'\n                    f\'First image: {previous["image"]}; label: {previous["label"]}; \'\n                    f\'other image: {image}; label: {label}. \'\n                    \'Compare annotations before resolving; do not select a copy arbitrarily.\')\n            duplicate = {\'removed\':str(image), \'kept\':previous[\'image\'], **comparison}\n            if comparison[\'annotation_match\']==\'bounded_rounding\':\n                duplicate.update({\'kept_boxes\':previous[\'boxes\'], \'removed_boxes\':boxes,\n                                  \'image_size\':size,\n                                  \'policy\':\'same class/count, one-to-one, <=1 pixel per corner and IoU >=0.99; canonical labels unchanged\'})\n            duplicates.append(duplicate)\n            continue\n        record = {\'image\': str(image), \'label\': str(label),\n                  \'source_split\': image.parent.parent.name,\n                  \'family\': source_family(image.stem), \'sha256_pixels\': exact,\n                  \'dhash\': dhash, \'size\': size, \'boxes\': boxes}\n        exact_seen[exact] = record\n        records.append(record)\n    return records, duplicates\n\n\ndef curate_export(root, names):\n    """Audit the whole export; exclude unresolved source families before splitting.\n\n    All copies of conflicting pixel groups and their source-family siblings are\n    excluded, including aliases in other pixel groups. Source files are read only.\n    No label merging, majority voting, or class relabeling occurs.\n    """\n    root=Path(root)\n    images=sorted(p for p in root.rglob(\'*\') if p.parent.name==\'images\'\n                  and p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.webp\'})\n    if not images:\n        raise ValueError(\'No images found in split/images folders.\')\n    pixels=defaultdict(list)\n    invalid=[]\n    bad_families=set()\n    bad_hashes=set()\n    for image in images:\n        label=image.parent.parent/\'labels\'/(image.stem+\'.txt\')\n        family=source_family(image.stem)\n        try:\n            exact,dhash,size=fingerprint(image)\n        except Exception as error:\n            invalid.append({\'image\':str(image),\'label\':str(label),\'family\':family,\n                            \'reason\':\'unreadable_image\',\'detail\':str(error)})\n            bad_families.add(family)\n            continue\n        try:\n            boxes=load_boxes(label,len(names))\n        except (ValueError,OSError,UnicodeError) as error:\n            invalid.append({\'image\':str(image),\'label\':str(label),\'family\':family,\n                            \'sha256_pixels\':exact,\'reason\':\'invalid_or_missing_annotation\',\n                            \'detail\':str(error)})\n            bad_families.add(family)\n            bad_hashes.add(exact)\n            continue\n        pixels[exact].append({\'image\':str(image),\'label\':str(label),\n            \'source_split\':image.parent.parent.name,\'family\':family,\n            \'sha256_pixels\':exact,\'dhash\':dhash,\'size\':size,\'boxes\':boxes})\n    conflicts=[]\n    for exact,copies in pixels.items():\n        # Pairwise comparison prevents a chain of small shifts exceeding the bound.\n        incompatible=any(compare_duplicate_annotations(a[\'boxes\'],b[\'boxes\'],a[\'size\']) is None\n                         for i,a in enumerate(copies) for b in copies[:i])\n        if incompatible:\n            conflicts.append({\'sha256_pixels\':exact,\n                \'reason\':\'annotation_difference_exceeds_bounded_rounding_or_class_count_mismatch\',\n                \'copies\':copies})\n            bad_hashes.add(exact)\n            bad_families.update(r[\'family\'] for r in copies)\n    # Transitively exclude every alias of a bad pixel group/source family.\n    changed=True\n    while changed:\n        changed=False\n        for exact,copies in pixels.items():\n            if exact in bad_hashes or any(r[\'family\'] in bad_families for r in copies):\n                before=(len(bad_families),len(bad_hashes))\n                bad_hashes.add(exact)\n                bad_families.update(r[\'family\'] for r in copies)\n                changed=changed or before!=(len(bad_families),len(bad_hashes))\n    records,duplicates,excluded=[],[],list(invalid)\n    for exact,copies in pixels.items():\n        if exact in bad_hashes:\n            excluded.extend(dict(r,reason=\'excluded_source_family_or_pixel_alias\') for r in copies)\n            continue\n        canonical=dict(copies[0],families=sorted({r[\'family\'] for r in copies}))\n        records.append(canonical)\n        for row in copies[1:]:\n            comparison=compare_duplicate_annotations(canonical[\'boxes\'],row[\'boxes\'],canonical[\'size\'])\n            duplicates.append({\'removed\':row[\'image\'],\'kept\':canonical[\'image\'],**comparison,\n                \'kept_boxes\':canonical[\'boxes\'],\'removed_boxes\':row[\'boxes\'],\'image_size\':canonical[\'size\']})\n    report={\'policy_version\':\'curated-v2\',\n        \'policy\':\'Exclude all unresolved pixel-group copies and source-family aliases before splitting; preserve source files and canonical labels.\',\n        \'rounding_policy\':{\'max_corner_delta_pixels\':1.,\'minimum_iou\':.99,\n                           \'same_class_and_count\':True,\'matching\':\'pairwise one-to-one\'},\n        \'source_images\':len(images),\'retained_unique_images\':len(records),\n        \'duplicate_copies_excluded\':len(duplicates),\'excluded_images\':len(excluded),\n        \'excluded_fraction\':len(excluded)/len(images),\'excluded_families\':sorted(bad_families),\n        \'invalid_files\':invalid,\'conflicting_pixel_groups\':conflicts,\n        \'excluded\':excluded,\'duplicates\':duplicates,\n        \'evaluation_scope\':\'Curated public subset; exclusions may change difficulty and class distribution. Local camera evaluation remains required.\'}\n    return records,report\n\n\ndef group_records(records, dhash_distance=6, physical_groups=None):\n    """Conservatively connect source siblings and near-duplicate image hashes.\n\n    A provided physical_groups mapping is authoritative but may be merged further\n    by similarity. Missing physical IDs are rejected, never invented.\n    """\n    parent = list(range(len(records)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    def union(i, j):\n        parent[find(j)] = find(i)\n    families = {}\n    physical = {}\n    for i, row in enumerate(records):\n        aliases=row.get(\'families\',[row[\'family\']])\n        for family in aliases:\n            if family in families:\n                union(i, families[family])\n            families[family] = i\n        if physical_groups is not None:\n            for key in aliases:\n                if key not in physical_groups or not physical_groups[key]:\n                    raise ValueError(f\'Missing physical specimen identity for {key}\')\n                specimen = physical_groups[key]\n                if specimen in physical:\n                    union(i, physical[specimen])\n                physical[specimen] = i\n    for i, first in enumerate(records):\n        for j in range(i):\n            second = records[j]\n            if (first[\'dhash\'] ^ second[\'dhash\']).bit_count() <= dhash_distance:\n                union(i, j)\n    grouped = defaultdict(list)\n    for i, row in enumerate(records):\n        grouped[find(i)].append(row)\n    return list(grouped.values())\n\n\ndef split_groups(groups, class_count, seed=42, attempts=500):\n    """Find a reproducible 60/20/20 group assignment with each class present.\n\n    If similarity collapses the data to too few independent groups, stop rather\n    than splitting those groups to manufacture a good-looking test result.\n    """\n    if len(groups) < 6:\n        raise ValueError(\'Fewer than 6 independent image groups: need more data or group review.\')\n    coverage={c:sum(any(b[0]==c for r in g for b in r[\'boxes\']) for g in groups)\n              for c in range(class_count)}\n    if any(n<3 for n in coverage.values()):\n        raise ValueError(f\'Each class needs at least 3 independent groups for train/val/test. Group coverage: {coverage}\')\n    best, best_score = None, float(\'inf\')\n    total_images = sum(map(len, groups))\n    totals = Counter(b[0] for g in groups for r in g for b in r[\'boxes\'])\n    rng = random.Random(seed)\n    for _ in range(attempts):\n        order = list(range(len(groups)))\n        rng.shuffle(order)\n        ntrain = max(1, int(len(order)*0.6))\n        nval = max(1, int(len(order)*0.2))\n        parts = {\'train\': order[:ntrain], \'val\': order[ntrain:ntrain+nval],\n                 \'test\': order[ntrain+nval:]}\n        if not all(parts.values()):\n            continue\n        score, valid = 0., True\n        for split, indexes in parts.items():\n            target = 0.6 if split == \'train\' else 0.2\n            rows = [r for index in indexes for r in groups[index]]\n            counts = Counter(b[0] for r in rows for b in r[\'boxes\'])\n            if any(counts[c] == 0 for c in range(class_count)):\n                valid = False\n                break\n            score += abs(len(rows)/total_images-target)\n            score += sum(abs(counts[c]/totals[c]-target) for c in range(class_count)) / class_count\n        if valid and score < best_score:\n            best, best_score = parts, score\n    if best is None:\n        raise ValueError(\'Cannot create group-disjoint splits containing every class. Collect/review groups.\')\n    result = {}\n    for split, indexes in best.items():\n        result[split] = []\n        for index in indexes:\n            for row in groups[index]:\n                result[split].append(dict(row, group_id=f\'group-{index:05d}\'))\n    return result\n\n\ndef materialize(splits, destination, names):\n    import yaml\n    destination = Path(destination)\n    if destination.exists():\n        raise ValueError(\'Prepared dataset already exists. Use a new experiment directory.\')\n    manifest, summary = [], {}\n    for split, rows in splits.items():\n        images = destination / \'images\' / split\n        labels = destination / \'labels\' / split\n        images.mkdir(parents=True)\n        labels.mkdir(parents=True)\n        counts = Counter()\n        for index, row in enumerate(rows):\n            stem = f\'{index:06d}_{row["sha256_pixels"][:12]}\'\n            image = images / (stem + Path(row[\'image\']).suffix.lower())\n            label = labels / (stem + \'.txt\')\n            shutil.copy2(row[\'image\'], image)\n            shutil.copy2(row[\'label\'], label)\n            counts.update(b[0] for b in row[\'boxes\'])\n            manifest.append({\'image\': str(image.relative_to(destination)),\n                             \'source_image\': str(row[\'image\']), \'split\': split,\n                             \'group_id\': row[\'group_id\'], \'family\': row[\'family\'],\n                             \'source_family_aliases\':json.dumps(row.get(\'families\',[row[\'family\']])),\n                             \'sha256_pixels\': row[\'sha256_pixels\']})\n        summary[split] = {\'images\': len(rows), \'boxes\': {names[c]: counts[c] for c in names},\n                          \'groups\': len({r[\'group_id\'] for r in rows}),\n                          \'negative_images\': sum(not r[\'boxes\'] for r in rows)}\n    with (destination/\'split_manifest.csv\').open(\'w\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(manifest[0]))\n        writer.writeheader()\n        writer.writerows(manifest)\n    config = {\'path\': str(destination.resolve()), \'train\': \'images/train\',\n              \'val\': \'images/val\', \'test\': \'images/test\', \'nc\': len(names), \'names\': names}\n    (destination/\'data.yaml\').write_text(yaml.safe_dump(config, sort_keys=False))\n    write_json(destination/\'audit.json\', summary)\n    return destination/\'data.yaml\', summary\n\n\ndef preview(root, split, names, output, limit=8):\n    from PIL import Image\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Rectangle\n    files = sorted((Path(root)/\'images\'/split).iterdir())\n    random.Random(42).shuffle(files)\n    columns = 4\n    fig, axes = plt.subplots(math.ceil(limit/columns), columns, figsize=(18, 9), squeeze=False)\n    for ax in axes.flat:\n        ax.axis(\'off\')\n    for ax, path in zip(axes.flat, files[:limit]):\n        with Image.open(path) as im:\n            rgb = im.convert(\'RGB\')\n            width, height = rgb.size\n            ax.imshow(rgb)\n        boxes = load_boxes(Path(root)/\'labels\'/split/(path.stem+\'.txt\'), len(names))\n        for c,x,y,w,h in boxes:\n            color = \'cyan\' if names[c] == \'strip\' else \'red\'\n            left, top = (x-w/2)*width, (y-h/2)*height\n            ax.add_patch(Rectangle((left, top), w*width, h*height, fill=False, ec=color, lw=2))\n            ax.text(left, top, names[c], fontsize=8, color=\'white\', backgroundcolor=color)\n        ax.set_title(path.name[:20])\n    fig.suptitle(f\'Ground truth: {split}; cyan=context, red=defect\')\n    fig.tight_layout()\n    fig.savefig(output, dpi=130)\n    return fig\n\n\ndef xyxy(box, width, height):\n    c,x,y,w,h = box\n    return [int(c), (x-w/2)*width, (y-h/2)*height,\n            (x+w/2)*width, (y+h/2)*height]\n\n\ndef iou(first, second):\n    area = max(0, min(first[2], second[2])-max(first[0], second[0])) * max(\n        0, min(first[3], second[3])-max(first[1], second[1]))\n    union = max(0, first[2]-first[0])*max(0, first[3]-first[1]) + max(\n        0, second[2]-second[0])*max(0, second[3]-second[1])-area\n    return area/union if union else 0.\n\n\ndef operating_metrics(samples, class_id, threshold, minimum_iou=0.5):\n    tp=fp=fn=0\n    for sample in samples:\n        gt = [b[1:] for b in sample[\'truth\'] if int(b[0]) == class_id]\n        predictions = sorted((b for b in sample[\'predictions\']\n                              if int(b[0]) == class_id and b[1] >= threshold),\n                             key=lambda b: -b[1])\n        matched = set()\n        for prediction in predictions:\n            choices = [(iou(prediction[2:], box), j) for j,box in enumerate(gt) if j not in matched]\n            overlap,index = max(choices, default=(0., -1))\n            if index >= 0 and overlap >= minimum_iou:\n                tp += 1\n                matched.add(index)\n            else:\n                fp += 1\n        fn += len(gt)-len(matched)\n    precision = tp/(tp+fp) if tp+fp else 0.\n    recall = tp/(tp+fn) if tp+fn else 0.\n    f2 = 5*precision*recall/(4*precision+recall) if precision+recall else 0.\n    return {\'tp\':tp, \'fp\':fp, \'fn\':fn, \'support\':tp+fn,\n            \'precision\':precision, \'recall\':recall, \'f2\':f2}\n\n\ndef tune_thresholds(samples, names, target_recall=0.95):\n    selected = {}\n    for c,name in names.items():\n        rows = []\n        for k in range(1, 100):\n            threshold = k/100\n            rows.append(dict(operating_metrics(samples, c, threshold), threshold=threshold))\n        feasible = [r for r in rows if r[\'support\'] > 0 and r[\'recall\'] >= target_recall]\n        if feasible:\n            best = max(feasible, key=lambda r: (r[\'precision\'], r[\'f2\'], r[\'threshold\']))\n        else:\n            best = max(rows, key=lambda r: (r[\'f2\'], r[\'recall\'], r[\'threshold\']))\n        selected[name] = dict(best, class_id=c, target_recall_met=bool(feasible),\n                              role=\'context\' if name == \'strip\' else \'defect\')\n    return selected\n\n\ndef collect_predictions(model, root, split, names, imgsz=640, device=0, batch_size=8):\n    """Collect CPU records with explicitly bounded image-list batches."""\n    from PIL import Image\n    if not isinstance(batch_size, int) or batch_size < 1:\n        raise ValueError(\'batch_size must be a positive integer\')\n    files = sorted(p for p in (Path(root)/\'images\'/split).iterdir() if p.is_file())\n    samples = []\n    for start in range(0, len(files), batch_size):\n        chunk = files[start:start + batch_size]\n        for result in model.predict(source=[str(p) for p in chunk], stream=True,\n                                    batch=batch_size, imgsz=imgsz, device=device,\n                                    conf=0.001, iou=0.7, max_det=300, verbose=False):\n            image = Path(result.path)\n            with Image.open(image) as im:\n                width, height = im.size\n            truth = load_boxes(Path(root)/\'labels\'/split/(image.stem+\'.txt\'), len(names))\n            predictions = []\n            for box in result.boxes.data.cpu().tolist():\n                x1, y1, x2, y2, confidence, c = box\n                predictions.append([int(c), confidence, x1, y1, x2, y2])\n            samples.append({\'image\': str(image),\n                            \'truth\': [xyxy(b, width, height) for b in truth],\n                            \'predictions\': predictions})\n        print(f\'{split}: collected {len(samples)}/{len(files)} images\', flush=True)\n    if len(samples) != len(files):\n        raise RuntimeError(\'Prediction count differs from input image count\')\n    return samples\n\n\ndef evaluate_frozen(samples, names, thresholds):\n    per_class = {name:operating_metrics(samples,c,thresholds[name][\'threshold\'])\n                 for c,name in names.items()}\n    defects = {c for c,name in names.items() if name != \'strip\'}\n    false_pass=false_hold=defective=good=0\n    for sample in samples:\n        actual = any(b[0] in defects for b in sample[\'truth\'])\n        predicted = any(b[0] in defects and b[1] >= thresholds[names[b[0]]][\'threshold\']\n                        for b in sample[\'predictions\'])\n        defective += actual\n        good += not actual\n        false_pass += actual and not predicted\n        false_hold += not actual and predicted\n    return {\'per_class\':per_class,\n            \'image_level\': {\'defective_images\':defective,\'negative_images\':good,\n                            \'false_pass_images\':false_pass,\'false_hold_images\':false_hold,\n                            \'false_pass_rate\':false_pass/defective if defective else None,\n                            \'false_hold_rate\':false_hold/good if good else None},\n            \'unit_note\':\'Image-level metrics, not per-strip production measurements.\'}\n\n\ndef prepare_calibration(root, output, count=512):\n    source=Path(root)/\'images\'/\'train\'\n    files=sorted(source.iterdir())\n    random.Random(42).shuffle(files)\n    selected=files[:count]\n    Path(output).mkdir(parents=True, exist_ok=True)\n    manifest=[]\n    for p in selected:\n        shutil.copy2(p, Path(output)/p.name)\n        manifest.append({\'file\':p.name,\'sha256\':file_hash(p),\'origin\':\'train_only\'})\n    write_json(Path(output)/\'calibration_manifest.json\',manifest)\n    return len(selected)\n\n\ndef export_onnx(checkpoint, output, samples, imgsz=640):\n    """Legacy static exporter is explicit for PyTorch 2.11, opset 11 for Hailo recipe.\n\n    Validates raw prediction parity; final Hailo parse/NMS/quantized evaluation\n    still require the vendor compiler and actual device.\n    """\n    import copy\n    import numpy as np\n    import torch\n    import onnx\n    import onnxruntime as ort\n    from ultralytics import YOLO\n    from ultralytics.data.augment import LetterBox\n    import cv2\n    predictor = YOLO(str(checkpoint))\n    model = copy.deepcopy(predictor.model).cpu().float().eval()\n    model.fuse()\n    for module in model.modules():\n        if hasattr(module,\'export\'):\n            module.export=True\n            module.format=\'onnx\'\n            module.dynamic=False\n    example=torch.zeros(1,3,imgsz,imgsz)\n    with torch.inference_mode():\n        expected=model(example)\n    if isinstance(expected,(tuple,list)):\n        raise ValueError(\'Expected one raw detection tensor; inspect the export wrapper.\')\n    output=Path(output)\n    torch.onnx.export(model, example, str(output), input_names=[\'images\'],\n                      output_names=[\'predictions\'], opset_version=11,\n                      dynamic_axes=None, dynamo=False)\n    graph=onnx.load(str(output))\n    onnx.checker.check_model(graph)\n    session=ort.InferenceSession(str(output),providers=[\'CPUExecutionProvider\'])\n    differences=[]\n    for file in samples:\n        image=cv2.imread(str(file))\n        if image is None:\n            raise ValueError(f\'Unreadable export validation image: {file}\')\n        letterbox=LetterBox(new_shape=(imgsz,imgsz), auto=False, stride=32)\n        image=letterbox(image=image)\n        array=np.ascontiguousarray(image[:,:,::-1].transpose(2,0,1))[None].astype(np.float32)/255.\n        with torch.inference_mode():\n            reference=model(torch.from_numpy(array)).numpy()\n        actual=session.run(None,{\'images\':array})[0]\n        np.testing.assert_allclose(actual,reference,rtol=1e-3,atol=1e-2)\n        differences.append({\'image\':Path(file).name,\'max_abs\':float(np.abs(actual-reference).max())})\n    inventory = [{\'name\':node.name,\'op\':node.op_type,\'outputs\':list(node.output)}\n                 for node in graph.graph.node]\n    write_json(output.with_suffix(\'.nodes.json\'),inventory)\n    return {\'status\':\'fp32_raw_tensor_parity_passed\',\'path\':str(output),\n            \'validation_images\':differences,\'shape\':[int(v) for v in expected.shape],\n            \'opset\':11,\'hailo_compatibility\':\'unverified\'}\n'
helper_file=WORK/'blistertrace_final.py'
helper_file.write_text(HELPER_SOURCE)
sys.path.insert(0,str(WORK))
import blistertrace_final as bt
print('Helpers ready.')


## Step 3 — Frozen experiment configuration
The main model is YOLOv8s, initialized from official COCO weights, not from the single-class starter. Early stopping can finish before the maximum epoch count. A future YOLOv8n comparison must use these same prepared splits in a separately tracked experiment.

Physical-group CSV is optional for public-source experiments, but required before claiming a physically independent local test: columns `family,group_id`, where family is the exported filename before `.rf.<hash>`. Without it, only source-image/perceptual separation is established.

Targets below are proposed acceptance targets, not expected or promised performance. Augmentation is modest; color shifts, blur, grayscale, mosaic and cutout are excluded from the final recipe.


In [ ]:
CONFIG={
 'experiment':'defects_v1', 'workspace':'blister-strips',
 'project':'blister-strips-defect-detection', 'version':2,
 'model':'yolov8s.pt', 'imgsz':640, 'epochs':100, 'patience':15,
 'batch':8, 'seed':42, 'dhash_distance':6,
 'target_recall':0.95, 'target_precision':0.80,
 'minimum_test_instances_per_defect':20, 'calibration_count':512
}
DATA_REVIEWED=False
PHYSICAL_GROUP_CSV=''  # optional path inside Kaggle, not a secret
EXP=WORK/CONFIG['experiment']
EXP.mkdir(exist_ok=True)
print(json.dumps(CONFIG,indent=2))
print('Training enabled after review:',DATA_REVIEWED)


## Step 4 — Download real defect data directly inside Kaggle
No image ZIP passes through your laptop. The SDK is used only for downloading public training data, not cloud inference. Its console output and exception messages are suppressed here to avoid exposing signed links or API credentials.

If the specified version is inaccessible, stop and share the sanitized error; do not switch to the pill-only starter. The dataset page lists its license as CC BY 4.0; retain the downloaded attribution files and inspect them before distribution.


In [ ]:
import contextlib, io, importlib.metadata
RAW=EXP/'source'
source_identity={k:CONFIG[k] for k in ('workspace','project','version')}
source_stamp=EXP/'source_identity.json'
if source_stamp.is_file():
    assert json.loads(source_stamp.read_text())==source_identity, 'Source config changed: choose a new experiment.'
if not (RAW/'data.yaml').is_file():
    from kaggle_secrets import UserSecretsClient
    try:
        api_key=UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception:
        raise RuntimeError('Add and enable the Kaggle Secret ROBOFLOW_API_KEY, then rerun this cell.') from None
    try:
        import roboflow
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            rf=roboflow.Roboflow(api_key=api_key)
            version=rf.workspace(CONFIG['workspace']).project(CONFIG['project']).version(CONFIG['version'])
            downloaded=version.download('yolov8',location=str(RAW),overwrite=False)
    except ModuleNotFoundError as error:
        missing=error.name or 'unknown'
        if not missing.replace('_','').replace('.','').isalnum(): missing='unknown'
        raise RuntimeError('Dataset SDK dependency missing: '+missing+
          '. Rerun the updated Step 2. No credential printed.') from None
    except Exception as error:
        raise RuntimeError('Dataset download failed ('+type(error).__name__+
          '). Check Internet, secret access and the public project/version. No credential printed.') from None
    finally:
        api_key=None
        rf=None
        version=None
    bt.write_json(source_stamp,source_identity)
assert (RAW/'data.yaml').is_file(), 'Dataset export layout differs; inspect downloaded files.'
print('Downloaded to:',RAW)
print('Source:', 'https://universe.roboflow.com/'+CONFIG['workspace']+'/'+CONFIG['project'])


## Step 5 — Audit, group related images, and construct held-out splits
The public export may contain augmented siblings. We remove exact duplicate images, group Roboflow source-name siblings, and conservatively connect near duplicates using a 256-bit image hash. Whole connected groups are assigned to train/val/test approximately 60/20/20 with class coverage.

This deliberately creates a **new grouped split** rather than reproducing the publisher's benchmark. Validation/test may still contain augmented images, and source-image grouping cannot prove physical-strip independence. Original physical specimen IDs or an independently captured local test are still needed for that claim.

Identical decoded images may have export rounding differences: accept only same-class, same-count, one-to-one boxes with every corner within one pixel and IoU at least 0.99, checked pairwise. Keep the first annotation unchanged and record both versions. Unresolved conflicting pixel groups, unreadable images, missing/invalid annotations, and all their source-family aliases are excluded from every split and recorded in `curation_audit.json`. Source files are preserved. Do not merge labels or guess the correct annotation. This creates a curated public subset; exclusions may change difficulty and class balance and must be reported.

If grouping leaves too few independent groups or insufficient class coverage, the notebook stops with a dataset-readiness error. Do not weaken the grouping merely to obtain high metrics.


In [ ]:
import yaml, csv
source_yaml=yaml.safe_load((RAW/'data.yaml').read_text())
raw_names=source_yaml['names']
if isinstance(raw_names,list): raw_names=dict(enumerate(raw_names))
NAMES={int(k):str(v).strip().lower() for k,v in raw_names.items()}
assert sorted(NAMES)==list(range(len(NAMES)))
assert len(set(NAMES.values()))==len(NAMES), 'Duplicate class names in source taxonomy.'
assert set(NAMES.values()) in ({'broken','cracked','missing'},
                             {'broken','cracked','missing','strip'}), (
    'Unexpected source taxonomy; review labels before training: '+str(NAMES))
records,curation=bt.curate_export(RAW,NAMES)
duplicates=curation['duplicates']
bt.write_json(EXP/'curation_audit.json',curation)
bt.write_json(EXP/'duplicate_audit.json',duplicates)
print('Source images:',curation['source_images'],'| Retained unique:',len(records),
      '| Excluded uncertain/invalid:',curation['excluded_images'])
print('Exact-pixel duplicates excluded:',len(duplicates),
      '| Bounded annotation rounding:',sum(d['annotation_match']=='bounded_rounding' for d in duplicates))
assert len(records)>=100, 'Unexpectedly small export; inspect source.'
physical=None
if PHYSICAL_GROUP_CSV:
    with open(PHYSICAL_GROUP_CSV,newline='') as stream:
        physical={row['family']:row['group_id'] for row in csv.DictReader(stream)}
groups=bt.group_records(records,CONFIG['dhash_distance'],physical)
print('Independent groups per class:',{name:sum(any(b[0]==c for r in g for b in r['boxes']) for g in groups) for c,name in NAMES.items()})
splits=bt.split_groups(groups,len(NAMES),CONFIG['seed'])
PREPARED=EXP/'prepared_clean_v2'
AUDIT_ID={
 'seed':CONFIG['seed'],'dhash_distance':CONFIG['dhash_distance'],
 'source_identity':source_identity,'classes':NAMES,
 'duplicate_policy':'same class/count, one-to-one, <=1 pixel per corner, IoU >=0.99',
 'curation_policy':curation['policy_version'],
 'excluded_families':curation['excluded_families'],
 'source_pixels':sorted(r['sha256_pixels'] for r in records),
 'source_labels':sorted((r['sha256_pixels'],r['boxes']) for r in records),
 'physical_groups':physical
}
audit_digest=__import__('hashlib').sha256(json.dumps(AUDIT_ID,sort_keys=True).encode()).hexdigest()
if PREPARED.exists():
    assert json.loads((EXP/'audit_identity_clean_v2.json').read_text())['sha256']==audit_digest, (
        'Data/grouping settings changed: use a new experiment.')
    DATA_YAML=PREPARED/'data.yaml'
    summary=json.loads((PREPARED/'audit.json').read_text())
else:
    DATA_YAML,summary=bt.materialize(splits,PREPARED,NAMES)
    bt.write_json(EXP/'audit_identity_clean_v2.json',{'sha256':audit_digest})
bt.write_json(EXP/'source_audit.json',{
 'unique_images':len(records),'exact_duplicates_removed':duplicates,
 'connected_groups':len(groups),'physical_group_ids_supplied':physical is not None,
 'group_size_distribution':sorted(len(g) for g in groups),
 'split_summary':summary,
 'curation_policy':curation['policy_version'],'excluded_images':curation['excluded_images'],
 'independence_limit':'Public source-image grouping; physical/production generalization unverified unless separately demonstrated.'})
print('Classes:',NAMES)
print('Unique images:',len(records),'| Connected groups:',len(groups))
print(json.dumps(summary,indent=2))
print('Unsupported classes: discoloration, unexpected contents, foreign particle, foil damage.')
print('No negative images in a split means false-hold specificity cannot be estimated there.')


## Step 6 — Defect annotation review
Check red defect boxes and cyan strip boxes if the export includes strip annotations. Inspect missing cavities, cracks, chips, class balance and small defects. This is the final mandatory human checkpoint before a long run. If the source labels are unsuitable, correct the data rather than accepting the preview.

After review set DATA_REVIEWED=True in Step 3, rerun Step 3, and continue from Step 7. Do not change the experiment name or training settings mid-run.


In [ ]:
import matplotlib.pyplot as plt
bt.preview(PREPARED,'train',NAMES,EXP/'annotation_preview.jpg')
plt.show()
print('PAUSE: confirm these are actual defect labels, not pill-presence boxes.')


## Step 7 — Full fine-tuning with reproducibility and recovery
Only completed runs with matching fingerprints are reused. Interrupted runs resume last.pt when its saved epoch and optimizer state support resumption. Completed best.pt is not an optimizer-resume checkpoint. A changed configuration needs a new experiment.

Albumentations' implicit default blur/grayscale transforms are explicitly disabled in this process. This avoids color/texture-changing augmentations being injected despite conservative trainer settings.


In [ ]:
assert DATA_REVIEWED, 'Review Step 6, enable DATA_REVIEWED in Step 3 and rerun Step 3.'
from ultralytics.data.augment import Albumentations
def no_implicit_albumentations(self,*args,**kwargs):
    self.p=0.0
    self.transform=None
    self.contains_spatial=False
Albumentations.__init__=no_implicit_albumentations
os.environ['WANDB_MODE']='disabled'
train_identity={'config':CONFIG,'audit_sha256':audit_digest,'ultralytics':ultralytics.__version__}
train_stamp=EXP/'train_identity.json'
if train_stamp.is_file():
    assert json.loads(train_stamp.read_text())==train_identity, 'Training config changed: use a new experiment.'
else:
    bt.write_json(train_stamp,train_identity)
RUN_DIR=EXP/'training'/'yolov8s'
BEST=RUN_DIR/'weights'/'best.pt'
COMPLETE=EXP/'training_complete.json'
if COMPLETE.is_file():
    saved=json.loads(COMPLETE.read_text())
    assert BEST.is_file() and bt.file_hash(BEST)==saved['best_sha256']
    print('Reusing completed verified training run:',BEST)
else:
    last=RUN_DIR/'weights'/'last.pt'
    if last.is_file():
        # Trusted checkpoint produced by this notebook, never an arbitrary URL.
        checkpoint=torch.load(last,map_location='cpu',weights_only=False)
        saved_epoch=int(checkpoint.get('epoch',-1))
        if saved_epoch<0 or checkpoint.get('optimizer') is None:
            assert BEST.is_file(), 'Checkpoint is stripped and best.pt is missing.'
            print('Completed/stripped checkpoint detected; retaining best.pt without pretending to resume optimizer state.')
        else:
            print('Resuming interrupted run from its local last.pt.')
            YOLO(str(last)).train(resume=True,device=0,workers=2)
    else:
        assert not RUN_DIR.exists(), 'Incomplete run without checkpoint: choose a new experiment.'
        YOLO(CONFIG['model']).train(
            data=str(DATA_YAML),imgsz=CONFIG['imgsz'],epochs=CONFIG['epochs'],
            patience=CONFIG['patience'],batch=CONFIG['batch'],device=0,workers=2,
            seed=CONFIG['seed'],deterministic=True,optimizer='AdamW',lr0=0.001,
            weight_decay=0.0005,warmup_epochs=3,cos_lr=True,
            hsv_h=0.,hsv_s=0.,hsv_v=0.15,degrees=8.,translate=0.05,scale=0.10,
            fliplr=0.,flipud=0.,mosaic=0.,mixup=0.,copy_paste=0.,
            amp=True,cache=False,save=True,save_period=10,plots=True,
            project=str(EXP/'training'),name='yolov8s',exist_ok=False)
    assert BEST.is_file()
    bt.write_json(COMPLETE,{'best_sha256':bt.file_hash(BEST)})
print('Candidate weights:',BEST)


## Step 8 — Validation metrics and operating thresholds
Collect validation detections at a low floor, then select each class's threshold on validation data only. Among thresholds meeting proposed recall, prefer precision; otherwise select best F2 and explicitly record the missed target.

`strip` is a context class: its confidence must not create a defect alert. Detection confidence is not severity. Class-absent/low-support results never become perfect scores. This simple matching uses same-class, one-to-one IoU ≥0.5; mAP remains Ultralytics' own metric.


In [ ]:
assert COMPLETE.is_file()
trained=YOLO(str(BEST))
validation=trained.val(data=str(DATA_YAML),split='val',imgsz=CONFIG['imgsz'],
    device=0,batch=CONFIG['batch'],workers=2,plots=True,
    project=str(EXP/'evaluation'),name='validation')
val_samples=bt.collect_predictions(trained,PREPARED,'val',NAMES,CONFIG['imgsz'])
thresholds=bt.tune_thresholds(val_samples,NAMES,CONFIG['target_recall'])
bt.write_json(EXP/'thresholds.json',thresholds)
bt.write_json(EXP/'validation_predictions.json',val_samples)
bt.write_json(EXP/'validation_metrics.json',{
 'mAP':validation.results_dict,'per_class':validation.summary(),
 'operating_point':bt.evaluate_frozen(val_samples,NAMES,thresholds),
 'threshold_selection':'Validation only; precision subject to recall target, else max F2.'})
print(json.dumps(thresholds,indent=2))


## Step 9 — Freeze before held-out test evaluation
This records a checkpoint/threshold/split fingerprint. Re-evaluating that exact candidate is permitted; changing it after inspecting test results is a new experiment requiring fresh final test data. This public grouped test is not a replacement for a local camera test.


In [ ]:
frozen={'best_sha256':bt.file_hash(BEST),'thresholds':thresholds,'split_sha256':audit_digest}
frozen_path=EXP/'frozen_evaluation.json'
if frozen_path.is_file():
    assert json.loads(frozen_path.read_text())==frozen, 'Candidate changed after test freeze.'
else:
    bt.write_json(frozen_path,frozen)
test_metrics=trained.val(data=str(DATA_YAML),split='test',imgsz=CONFIG['imgsz'],
    device=0,batch=CONFIG['batch'],workers=2,plots=True,
    project=str(EXP/'evaluation'),name='heldout_test')
test_samples=bt.collect_predictions(trained,PREPARED,'test',NAMES,CONFIG['imgsz'])
test_operating=bt.evaluate_frozen(test_samples,NAMES,thresholds)
bt.write_json(EXP/'test_predictions.json',test_samples)
bt.write_json(EXP/'test_metrics.json',{
 'mAP':test_metrics.results_dict,'per_class':test_metrics.summary(),
 'operating_point':test_operating,
 'scope':'Curated public subset excluding unresolved conflict/invalid families; source-family/perceptually grouped split; not the publishers benchmark or proof of physical independence.'})
print(json.dumps(test_operating,indent=2))


## Step 10 — Review errors and image-level decisions
Produce overlays at frozen class thresholds. Missing a defect may produce a false pass; inspect the saved error list. A no-defect image is not automatically an approved pharmaceutical pack. A production PASS needs alignment/quality/coverage checks and a validated inspection profile.


In [ ]:
from PIL import Image,ImageDraw
from IPython.display import display
ERRORS=EXP/'error_review'
ERRORS.mkdir(exist_ok=True)
error_list=[]
ranked=[]
for sample in test_samples:
    item=bt.evaluate_frozen([sample],NAMES,thresholds)
    wrong=sum(r['fp']+r['fn'] for name,r in item['per_class'].items() if name!='strip')
    ranked.append((wrong,sample))
    if wrong: error_list.append({'image':sample['image'],'metrics':item})
for index,(wrong,sample) in enumerate(sorted(ranked,key=lambda x:-x[0])[:12]):
    with Image.open(sample['image']) as im: canvas=im.convert('RGB')
    draw=ImageDraw.Draw(canvas)
    for c,confidence,x1,y1,x2,y2 in sample['predictions']:
        name=NAMES[c]
        if confidence>=thresholds[name]['threshold']:
            color='cyan' if name=='strip' else 'red'
            draw.rectangle([x1,y1,x2,y2],outline=color,width=3)
            draw.text((x1,max(0,y1-12)),f'{name} {confidence:.2f}',fill=color)
    canvas.save(ERRORS/f'{index:02d}_errors_{wrong}.jpg')
    if index<4: display(canvas)
bt.write_json(EXP/'error_review.json',error_list)
print('Test images with defect localization/classification errors:',len(error_list))


## Step 11 — Static FP32 ONNX and numerical parity
Export the selected detector using an explicit static legacy ONNX path, then compare raw outputs on eight **validation** images against PyTorch FP32. This avoids mixing test data into export adjustments. The raw output excludes NMS.

ONNX structural validity is insufficient: parity must pass. If export fails, the notebook still packages best.pt and its error status, but marks export incomplete. Do not install ExecuTorch into this training environment to work around it. The actual Hailo parser still needs to accept the head graph and correct custom class count.


In [ ]:
EXPORT_DIR=EXP/'export'
EXPORT_DIR.mkdir(exist_ok=True)
try:
    samples=sorted((PREPARED/'images'/'val').iterdir())[:8]
    export_status=bt.export_onnx(BEST,EXPORT_DIR/'blistertrace_yolov8s.onnx',samples,CONFIG['imgsz'])
except Exception as error:
    export_status={'status':'failed','error':str(error),'hailo_compatibility':'unverified'}
    print('EXPORT INCOMPLETE. Training weights retained:',error)
bt.write_json(EXP/'export_status.json',export_status)
print(json.dumps(export_status,indent=2))


## Step 12 — Training-only calibration sample and portable model contract
Calibration uses only the training split. The requested 512 is a bounded prototype set, not a guarantee that every Hailo optimization configuration has enough examples. Inspect the compiler's requirements; collect more representative training-side images when needed. No duplicated or test images are manufactured to reach a count.

Hailo may own input normalization and NMS. Confirm that ownership when compiling; do not normalize twice. Raw PyTorch/ONNX expects normalized RGB float32 NCHW; the Hailo runtime input contract must be separately verified.


In [ ]:
CALIBRATION=EXP/'calibration_train_only'
calib_count=bt.prepare_calibration(PREPARED,CALIBRATION,CONFIG['calibration_count'])
contract={
 'model':'YOLOv8s custom blister defects','classes':NAMES,
 'defect_classes':[name for name in NAMES.values() if name!='strip'],
 'context_classes':[name for name in NAMES.values() if name=='strip'],
 'input':{'shape':[1,3,CONFIG['imgsz'],CONFIG['imgsz']],
   'dtype':'float32','layout':'NCHW','color':'RGB','range':[0,1],
   'resize':'Aspect-preserving letterbox; pad RGB 114; preserve scale and padding.',
   'calibration_note':'Follow Hailo model script normalization ownership, not this tensor contract blindly.'},
 'output':{'raw_shape':export_status.get('shape'),'nms_embedded':False,
   'description':'YOLOv8 decoded raw boxes/scores, model-specific head; inspect exported tensor contract.',
   'postprocess':'class-aware NMS at IoU 0.7; frozen per-class confidence thresholds; map coordinates back.'},
 'severity_policy_demo':{'missing':'major','broken':'major','cracked':'major'},
 'decision_note':'Context strip is not a defect; absent detections alone do not establish complete PASS.',
 'target':{'vision':'Hailo-8 / HailoRT after compile and INT8 validation',
           'advisory':'Separate Llama-family .pte / ExecuTorch / Pi Arm CPU'},
 'unsupported':['discoloration','unexpected contents','foreign particle','foil damage'],
 'calibration_images':calib_count,
 'training_source':'Roboflow Blister strips / version '+str(CONFIG['version'])
}
bt.write_json(EXP/'model_spec.json',contract)
print('Training-only calibration images:',calib_count)
print('Hailo INT8 compilation and validation remain pending.')


## Step 13 — Honest readiness report and artifact bundle
The report checks proposed targets per defect rather than quoting an aggregate context-dominated mAP. Failed/low-support results remain visible. A physically independent local camera test, Hailo runtime validation, and mandatory Llama/ExecuTorch evidence are always separate requirements.

The ZIP contains weights, ONNX if valid, calibration images, exact dependency list, model/threshold contracts, evaluation, previews and attribution. It excludes the full source dataset. No API key is saved. Download this ZIP and Save Version with outputs before ending the Kaggle session.


In [ ]:
checks={}
for name,row in test_operating['per_class'].items():
    if name=='strip': continue
    checks[name]={
      'support_sufficient':row['support']>=CONFIG['minimum_test_instances_per_defect'],
      'recall_target_met':row['recall']>=CONFIG['target_recall'],
      'precision_target_met':row['precision']>=CONFIG['target_precision']}
readiness={
 'test_targets':checks,'fp32_export':export_status['status'],
 'physical_group_ids_supplied':physical is not None,
 'negative_test_images':test_operating['image_level']['negative_images'],
 'hailo_int8_parity':'pending','pi_performance':'pending',
 'local_camera_generalization':'pending','llama_executorch_advisory':'pending',
 'full_ps5_submission_ready':False,
 'note':'This bundle is a measured vision candidate, not proof of production readiness or complete PS5 compliance.'}
bt.write_json(EXP/'readiness.json',readiness)
import shutil
BUNDLE=EXP/'submission_vision_bundle'
BUNDLE.mkdir(exist_ok=True)
for filename in ('environment.txt','blistertrace_final.py'):
    shutil.copy2(WORK/filename,BUNDLE/filename)
for p in EXP.glob('*.json'): shutil.copy2(p,BUNDLE/p.name)
for p in EXP.glob('*.jpg'): shutil.copy2(p,BUNDLE/p.name)
for p in (PREPARED/'split_manifest.csv', PREPARED/'data.yaml', PREPARED/'audit.json'):
    shutil.copy2(p,BUNDLE/p.name)
folders=[RUN_DIR,CALIBRATION,ERRORS]
if export_status['status']=='fp32_raw_tensor_parity_passed': folders.append(EXPORT_DIR)
for folder in folders:
    shutil.copytree(folder,BUNDLE/folder.name,dirs_exist_ok=True)
for p in RAW.glob('README*'):
    if p.is_file(): shutil.copy2(p,BUNDLE/('source_'+p.name))
bt.write_json(BUNDLE/'source_attribution.json',{
 'title':"Blister Strips defect detection",'author':'Blister strips',
 'url':'https://universe.roboflow.com/blister-strips/blister-strips-defect-detection',
 'license_as_listed':'CC BY 4.0','version':CONFIG['version'],
 'changes':'Bounded-rounding deduplication; unresolved conflict/invalid source families excluded; source-family alias/perceptual grouping; new held-out splits; online training augmentation.'})
(BUNDLE/'README.txt').write_text(chr(10).join([
    'Vision candidate: broken/cracked/missing; optional strip context only if declared in model_spec.json. Other defects unsupported.',
    'Do not present public split results as physical Pi performance or complete PS5 compliance.',
    'Hailo requires compatible vendor compilation, calibration and INT8 quality checks.',
    'Llama via ExecuTorch on Pi Arm CPU is a separate mandatory pipeline.']))
bt.write_json(BUNDLE/'file_hashes.json',{
    str(p.relative_to(BUNDLE)):bt.file_hash(p) for p in BUNDLE.rglob('*') if p.is_file() and p.name!='file_hashes.json'})
zip_path=shutil.make_archive(str(EXP/'BlisterTrace_Submission_Vision'),'zip',root_dir=BUNDLE)
print(json.dumps(readiness,indent=2))
print('Download bundle:',zip_path)
print('SHA256:',bt.file_hash(zip_path))
from IPython.display import FileLink
display(FileLink(str(Path(zip_path).relative_to(Path('/kaggle/working')))))


## Remaining deployment and mandatory advisory work

### Hailo-8 vision branch
- Use a compatible **Model Zoo 2.x / DFC 3.x** release on a supported Linux x86-64 compiler host.
- The Hailo v2.17 YOLOv8 training guide documents pretrained YOLOv8s, static opset-11 ONNX and custom HEF compilation. The model's exact parser endpoints must be verified from its node inventory; no invented node flags are provided here.
- Supply the correct custom class count and class order. Verify normalization and letterbox behavior, Hailo NMS configuration, and target Hailo-8 architecture. Preserve a compiler/environment manifest.
- Compare FP32 versus INT8 detection metrics and per-class recall using frozen data. Recalibrate operating thresholds on validation if quantization changes scores; freeze again before final testing.
- Test HailoRT, camera input, alert latency, CPU/RAM/temperature and sustained operation on the actual Pi. Host GPU speed is not a substitute.

### Separate mandatory Llama branch
This notebook does not train or deploy Llama. Use Llama 3.2 1B Instruct with a coherent ExecuTorch export/runtime version, tokenizer, supported XNNPACK quantization and KV cache, as described in the course's linked LLM material. Keep that environment separate from Kaggle's CUDA training packages.

Llama receives immutable inspection facts and retrieved local SOP clauses; rules decide disposition/severity. Validate its advisory JSON, cite clauses, draft QA notes, and provide a labeled deterministic fallback. Demonstrate the actual .pte running through ExecuTorch on the Pi Arm CPU. An ONNX detector cannot satisfy that requirement.

### Course reuse map
| Course material | Use in this workflow |
|---|---|
| Module 0 | AOT preparation versus runtime, backend-specific artifacts |
| Module 1 | Explicit model contract, export, target validation, separate LLM follow-on |
| Module 2 | FP32 baseline, representative calibration, quality/latency trade-offs |
| Module 2 optional YOLO | Letterbox, class scores, NMS, correct coordinate restoration |
| Module 2 optional profiling | ETRecord/ETDump/Inspector if CPU runtime investigation is needed later |
| Module 3 | Input/output consistency, app responsibilities, UI responsiveness |
| Module 4 | Hardware-specific compiler and fallback lessons; **not** a Hailo recipe |

## Primary references
- [Dataset scope and license](https://universe.roboflow.com/blister-strips/blister-strips-defect-detection)
- [Roboflow SDK download API](https://docs.roboflow.com/reference/platform/python-sdk.md)
- [Hailo-8 YOLOv8 recipe, v2.17](https://raw.githubusercontent.com/hailo-ai/hailo_model_zoo/v2.17/training/yolov8/README.rst)
- [Hailo compatibility](https://github.com/hailo-ai/hailo_model_zoo)
- [Training API](https://docs.ultralytics.com/modes/train/)
- [PyTorch ONNX API](https://docs.pytorch.org/docs/2.11/onnx.html)
- [Arm course](https://learn.arm.com/courses/executorch-on-arm/)
- [ExecuTorch LLM export](https://docs.pytorch.org/executorch/stable/llm/export-llm.html)

Keep source data attribution, model and package licenses (including Ultralytics AGPL-3.0). No hosted inference is needed for the deployed project. Data downloading and training happen in the development phase.
